# Phase 12a — two retrieval fixes: clause merging and query translation

Two concrete weaknesses came out of the Phase 8 failure analysis, and diagnosing them pointed at two independent fixes.

## Problem 1 — one article can crowd out every other article

Long articles were split at their clauses during chunking, so a single article can occupy many of the 10 slots handed to the generator. Traced concretely on **q22** ("will my pension go up over the years?", ground truth SSL Article 90):

| slot | what was returned |
|---|---|
| 1, 2, 5, 6, 8, 9 | SSL Article 71 (six separate clauses, all saying "pension shall be increased by X%") |
| 7 | SSL Article 71 again, Arabic |
| 3, 10 | SSL Article 62 |
| 4 | SSL Article 85 |
| — | **Article 90 never appeared** — it sat at semantic rank 11, one place outside the cutoff |

So Article 90 wasn't ranked badly; it was squeezed out by near-duplicate clauses of an article already present seven times. Across the eval set, 5 of 40 questions lose ≥3 slots this way — and two of them (q22, q33) are retrieval failures, i.e. **half of all "never retrieved" cases trace to this single cause.**

Only 21 of 531 (law, article, language) units are split into multiple clauses at all, so this affects a small, identifiable set of articles — mostly the SSL pension provisions (71 is split into 29 pieces in English, 64 into 14, 85 into 15).

**Fix tested:** group the *retrieved* clauses by article and merge them into a single slot. Not "pick one clause" (which risks keeping clause (a) when the answer is in clause (d), scoring a hit while the generator gets useless context) and not "deliver the whole article" (which would dump 26 irrelevant clauses in alongside the 3 that matched). If three clauses of Article 71 rank highly, that is signal that three clauses matter — keep exactly those, merged, in one slot.

## Problem 2 — cross-lingual retrieval is structurally handicapped

Cross-lingual recall@1 has been 0.0 since Phase 4, against 0.844 same-language. Two independent reasons:
- The embedding model is strong within a language and weak across languages (a known Phase 4 finding, and the reason reranking was even considered in Phase 8c).
- BM25 contributes **nothing at all** across languages, by construction — an Arabic query and an English article share zero tokens, so the keyword half of the hybrid is dead weight on exactly these questions.

**Fix tested:** translate the question into the other language, retrieve with both versions, and fuse. This converts a cross-language matching problem into two same-language ones, which is where both rankers are strong. Cost is one small LLM call per question.

Both fixes are evaluated on the same 40-question eval set and the same metrics used for every other retrieval decision in this project. Kernel: `rag311`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT = Path(r"c:\Users\Mohannad Alsaleh\Desktop\Data Science\Data Science Projects\1st RAG project")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.retrieval import Retriever, RRF_K, RRF_W, CANDIDATE_TOP_K

retriever = Retriever()

eval_questions = pd.DataFrame([json.loads(l) for l in open(PROJECT / "eval" / "eval_draft.jsonl", encoding="utf-8")])
LAW_MAP = {"Work_Law_1996": "labor96", "Social_Security_Law": "ssl", "Professional_Work_Law_2019": "prof19"}
eval_questions["answer_law_code"] = eval_questions["answer_law"].map(LAW_MAP)
eval_questions["is_cross_lingual"] = eval_questions["question_language"] != eval_questions["answer_language"]
print(f"{len(eval_questions)} questions | {eval_questions.is_cross_lingual.sum()} cross-lingual")

## Building blocks

`fuse` is the existing RRF from `src/retrieval.py`, factored out so the same fusion can be applied to one query or to two (original + translated). `assemble` turns a ranked chunk list into the slots actually handed to the generator, either one-chunk-per-slot (current behaviour) or clauses-merged-per-article.

In [ ]:
def fuse(rank_lists):
    """RRF over (ranked_ids, weight) pairs -> ranked chunk_ids, best first."""
    scores = {}
    for ids, weight in rank_lists:
        for rank, cid in enumerate(ids, start=1):
            scores[cid] = scores.get(cid, 0.0) + weight / (RRF_K + rank)
    return [cid for cid, _ in sorted(scores.items(), key=lambda kv: -kv[1])]


def candidates_for(query):
    """The two base rankers, weighted as in production."""
    return [
        (retriever._semantic_candidates(query, CANDIDATE_TOP_K), RRF_W),
        (retriever._keyword_candidates(query, CANDIDATE_TOP_K), 1 - RRF_W),
    ]


def row_of(cid):
    return retriever.chunks.iloc[retriever.id_to_row[cid]]


def assemble(ranked_ids, top_k, merge_clauses):
    """Ranked chunk ids -> the slots given to the generator.

    merge_clauses=False reproduces current production behaviour: one chunk per slot.
    merge_clauses=True groups the retrieved clauses of an article into a single slot,
    keeping only the clauses that were actually retrieved, in document order.
    """
    if not merge_clauses:
        out = []
        for cid in ranked_ids[:top_k]:
            r = row_of(cid)
            out.append({"law": r["law"], "article_number": r["article_number"],
                        "language": r["language"], "clause": r["clause"], "text": r["text"]})
        return out

    groups, order = {}, []
    for cid in ranked_ids:
        r = row_of(cid)
        key = (r["law"], r["article_number"], r["language"])
        if key not in groups:
            if len(order) >= top_k:
                continue  # slot budget spent; ignore further new articles
            groups[key] = []
            order.append(key)
        groups[key].append(cid)

    out = []
    for key in order[:top_k]:
        # merge in document order, not retrieval order, so clauses read correctly
        cids = sorted(groups[key], key=lambda c: retriever.id_to_row[c])
        rows = [row_of(c) for c in cids]
        out.append({
            "law": key[0], "article_number": key[1], "language": key[2],
            "clause": ", ".join(r["clause"] for r in rows if r["clause"]),
            "text": "\n".join(r["text"] for r in rows),
            "n_clauses": len(rows),
        })
    return out


def rank_of_correct(slots, q):
    correct = set(q.answer_article_numbers)
    for pos, s in enumerate(slots, start=1):
        if s["law"] == q.answer_law_code and s["language"] == q.answer_language and s["article_number"] in correct:
            return pos
    return None


def metrics(df, col="rank"):
    n = len(df); r = df[col]
    return {"n": n,
            "recall@1": round((r == 1).sum() / n, 3),
            "recall@5": round((r.notna() & (r <= 5)).sum() / n, 3),
            "recall@10": round(r.notna().sum() / n, 3),
            "mrr": round((1 / r.dropna()).sum() / n, 3)}

print("ready")

## Query translation

GPT-4o-mini via OpenRouter (the same model already used for generation, so no new dependency). Translations are computed once and cached to `eval/query_translations.json` so re-running the notebook doesn't re-pay for them.

In [ ]:
import time

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(PROJECT / ".env")
client = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")

TRANSLATE_PROMPT = (
    "Translate this question about Jordanian labour / social security law into {target}. "
    "Use the standard legal terminology a lawyer would use, so the translation matches how "
    "the statute itself is worded. Reply with the translation only, nothing else.\n\n{q}"
)

CACHE = PROJECT / "eval" / "query_translations.json"
translations = json.loads(CACHE.read_text(encoding="utf-8")) if CACHE.exists() else {}

def translate(question, source_lang, max_retries=4):
    if question in translations:
        return translations[question]
    target = "English" if source_lang == "ar" else "Arabic"
    last = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="openai/gpt-4o-mini",
                messages=[{"role": "user", "content": TRANSLATE_PROMPT.format(target=target, q=question)}],
            )
            if resp.choices:
                out = (resp.choices[0].message.content or "").strip()
                if out:
                    translations[question] = out
                    return out
        except Exception as e:
            last = e
        time.sleep(min(20, 2 ** attempt))
    raise RuntimeError(f"translation failed: {last}")

for q in eval_questions.itertuples(index=False):
    translate(q.question_text, q.question_language)

CACHE.write_text(json.dumps(translations, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{len(translations)} translations cached")

sample = eval_questions.iloc[0]
print("\nexample:")
print("  original  :", sample.question_text)
print("  translated:", translations[sample.question_text])

## Evaluate the four combinations

Base rankers are computed once per query (original and translated) and reused across configurations, so the only repeated work is the fusion and assembly.

In [ ]:
cand_orig, cand_trans = {}, {}
for q in eval_questions.itertuples(index=False):
    cand_orig[q.question_id] = candidates_for(q.question_text)
    cand_trans[q.question_id] = candidates_for(translations[q.question_text])
print("candidates cached for", len(cand_orig), "questions")

CONFIGS = {
    "baseline (production)":        dict(translate=False, merge=False),
    "clause merging":               dict(translate=False, merge=True),
    "query translation":            dict(translate=True,  merge=False),
    "translation + clause merging": dict(translate=True,  merge=True),
}

rows = []
for name, cfg in CONFIGS.items():
    for q in eval_questions.itertuples(index=False):
        lists = list(cand_orig[q.question_id])
        if cfg["translate"]:
            # the translated query's rankers get the same relative weights, halved so the
            # two language views contribute equally rather than doubling total weight
            lists = [(ids, w / 2) for ids, w in lists]
            lists += [(ids, w / 2) for ids, w in cand_trans[q.question_id]]
        slots = assemble(fuse(lists), 10, cfg["merge"])
        rows.append({"config": name, "question_id": q.question_id,
                     "is_cross_lingual": q.is_cross_lingual, "rank": rank_of_correct(slots, q)})

results = pd.DataFrame(rows)
results.to_csv(PROJECT / "eval" / "retrieval_improvements.csv", index=False, encoding="utf-8-sig")

for name in CONFIGS:
    d = results[results.config == name]
    print(f"--- {name} ---")
    print("  overall      ", metrics(d))
    print("  same-language", metrics(d[~d.is_cross_lingual]))
    print("  cross-lingual", metrics(d[d.is_cross_lingual]))

## Which questions changed, and did anything regress?

In [ ]:
pivot = results.pivot(index="question_id", columns="config", values="rank")
pivot = pivot[list(CONFIGS)]
pivot["cross_lingual"] = results.groupby("question_id").is_cross_lingual.first()

base = pivot["baseline (production)"]
for name in list(CONFIGS)[1:]:
    col = pivot[name]
    better = ((base.isna() & col.notna()) | (col < base)).sum()
    worse = ((col.isna() & base.notna()) | (col > base)).sum()
    print(f"{name}: {better} questions improved, {worse} regressed")

print("\n--- questions where any config differs from baseline ---")
changed = pivot[pivot[list(CONFIGS)].nunique(axis=1, dropna=False) > 1]
changed

In [ ]:
# Clause merging changes how much text reaches the generator -- worth knowing before
# committing to it, since context size drives both cost and answer-dilution risk.
sizes = []
for merge in (False, True):
    for q in eval_questions.itertuples(index=False):
        slots = assemble(fuse(cand_orig[q.question_id]), 10, merge)
        sizes.append({"merge": merge, "question_id": q.question_id,
                      "slots": len(slots), "chars": sum(len(s["text"]) for s in slots)})
s = pd.DataFrame(sizes)
print(s.groupby("merge").agg(mean_slots=("slots", "mean"), mean_chars=("chars", "mean"),
                             median_chars=("chars", "median"), max_chars=("chars", "max")).round(0).to_string())